# HealthConnect Clinic - Week 4 Initial Analysis
### AnalystLab Arica Experience Lab | Data Analyst Track

**Central Project Question:** How can HealthConnect Clinic use data and AI to reduce missed appointments and improve the pateint support experience?

In [1]:
import pandas as pd
pd.set_option ('display.max_columns', None)

df = pd.read_csv('HealthConnect_Appointment_Data.csv')
print (f"Rows:{df.shape[0]}, Columns:{df.shape[1]}")
df.head()

Rows:5000, Columns:18


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   str    
 1   patient_id             5000 non-null   str    
 2   gender                 5000 non-null   str    
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   str    
 5   appointment_type       5000 non-null   str    
 6   booking_date           5000 non-null   str    
 7   appointment_date       5000 non-null   str    
 8   appointment_day        5000 non-null   str    
 9   appointment_time       5000 non-null   str    
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   str    
 14  reminder_channel       3634 non-null   str    
 15  distance_to_cli

In [4]:
missing = df.isnull().sum()
missing_pct = (missing/len(df) * 100).round(2)
pd.DataFrame({'Missing_count': missing, 'missing_pct': missing_pct})[missing > 0].sort_values('missing_pct', ascending=False)

,Missing_count,missing_pct
reminder_channel,1366,27.32
distance_to_clinic_km,90,1.80
waiting_time_minutes,60,1.20


### Observations on Missing Data

Looking at the missing values above, there are three columns with gaps:

- **reminder_channel** is missing for 1,366 rows. This is not a mistake in the data, it lines up exactly with the rows where `reminder_sent` is "No." In other words, if no reminder was sent, there's naturally no channel to record. This should be treated as "Not Applicable" rather than a data problem.

- **distance_to_clinic_km** is missing for 90 rows (about 1.8% of the data).

- **waiting_time_minutes** is missing for 60 rows (about 1.2% of the data).

These last two are small enough that they likely don't affect the overall analysis much, but I will decide later (in Week 5) whether to exclude these rows or fill in the missing values before calculating KPIs.

In [5]:
print("Duplicate Rows:", df.duplicated().sum())
print("Duplicate appointment_id:", df['appointment_id'].duplicated().sum())

Duplicate Rows: 0
Duplicate appointment_id: 0


### Duplicate Check

There are no duplicate rows and no duplicate appointment IDs in the dataset. This confirms each rows represents a unique, distinct appointment record.

In [6]:
df['booking_date_dt'] = pd.to_datetime(df['booking_date'], format='%m/%d/%Y')
df['appointment_date_dt'] = pd.to_datetime(df['appointment_date'], format='%m/%d/%Y')
df['calc_lead_days'] = (df['appointment_date_dt'] - df['booking_date_dt']).dt.days

checks = {
    'lead days match': (df['calc_lead_days'] == df['booking_lead_days']).all(),
    'no negative lead days': (df['calc_lead_days'] >= 0).all(),
    'weekday matches appointment_day': (df['appointment_date_dt'].dt.day_name() == df['appointment_day']).all(),
    'no_shows <= previous_appointments': (df['previous_no_shows'] <= df['previous_appointments']).all(),
}
for check, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", check)

PASS - lead days match
PASS - no negative lead days
PASS - weekday matches appointment_day
PASS - no_shows <= previous_appointments


### Logical Consistency Checks

All four checks passed:
- The booking lead time matches the actual gap between data and appointment date.
- No appointment has a negative lead time (i.e., nothing was booked after it already happened).
- The day of the week recorded matches the actual date.
- No patient has more previous no-shows than previous appointments (which wouldn't make sense).

This means the dataset is internally consistent and reliable to work with - there are no contradictions in the data.

In [7]:
outcome_counts = df['appointment_outcome'].value_counts()
outcome_pct = df['appointment_outcome'].value_counts(normalize=True).mul(100).round(1)
pd.DataFrame({'count': outcome_counts, 'pct': outcome_pct})

,count,pct
appointment_outcome,,
No-Show,2423,48.5
Attended,2314,46.3
Cancelled,263,5.3


### Appointment Outcome Distribution

No-shows make up about 48.5%  of all appointments, attended appointsmet are about 46.3%, and cancellations are about 5.3%.

This is an important finding: almost half of all appointments end in a no-show. This confirms that missed appointments are a major, ongoing problem for HealthConnect Clinic, not a rare exception.

In [8]:
pd.crosstab(df['reminder_sent'], df['appointment_outcome'], normalize='index').mul(100).round(1)

appointment_outcome,Attended,Cancelled,No-Show
reminder_sent,,,
No,42.7,5.9,51.4
Yes,47.6,5.0,47.4


### Reminder and No-Shows

Patients who received a reminder had a no-show rate of about 47.3%, compared to 51.4% for patients who did not receive one. This suggest reminders may be linked to slightly better attendance, though this is just an early observation, not proof that reminders directly cause better attendance.

In [9]:
df['had_prev_noshow'] = df['previous_no_shows'] > 0
pd.crosstab(df['had_prev_noshow'], df['appointment_outcome'], normalize='index').mul(100).round(1)

appointment_outcome,Attended,Cancelled,No-Show
had_prev_noshow,,,
False,50.5,6.0,43.5
True,40.4,4.2,55.4


### Prior No-Shows History and Future No-Shows

Patients who have missed an appointment before have a no-show rate of about 55.4%, compared to 43.5% for patients with no prior no-show history. This suggests that past behaviour is a strong signal for predicting future no-shows, and could be a useful factor for the clinic to watch going forward.

## Business Questions and Candidate KPIs

Based on what I've seen in the data so far, here are five business questions I think are worth investigating further, along with a KPI (measurement) linked to each one. At this stage I'm only identifying these KPIs; I'm not calculating or charting them yet, that will happen in Week 5.

**1. What is the overall no-show rate, and how does it vary across patient groups (age, gender), appointment types, and time slots?**
- KPI: Overall No-Show Rate, and No-Show Rate by Segment (age group, gender, appointment type, day/time)

**2. Does sending a reminder, and which channel is used (SMS, WhatsApp, Email) relate to whether a patient attends?**
- KPI: Reminder Effectiveness Rate (no-show rate compared between reminder sent vs. not sent, and by channel)

**3. How much does a patient's past no-show history predict whether they'll miss a future appointment?**
- KPI: Repeat No-Show Rate (no-show rate for patients with a prior no-show vs. those with none)

**4. Does how far in advance an appointment is booked relate to the chance of a no-show?**
- KPI: No-Show Rate by Booking Lead Time Band (e.g. grouped into 0-7 days, 8-30 days, 31-60 days)

**5. Do access-related factors- distance to the clinic and past waiting times- relate to no-show behavior?**
- KPI: No-Show Rate by Distance Band and by Waiting Time Band

Each of these KPIs can be calculated directly from the existing columns in the dataset, so no additional data is needed to start on them next.

## Initial Analysis Approach (Plan for Week 5)

My plan for the next phase of this project is:

1. **Calculate the KPIs above** for the whole dataset first, to get baseline numbers.
2. **Break each KPI down by segment** (age group, appointment type, day of week, time of day) to see where the no-show problem is concentrated.
3. **Look more closely at relationships** between no-shows and the key factors identified (reminders, prior no-show history, booking lead time, distance, waiting time) using simple comparisons and charts.
4. **Decide how to handle cancelled appointments** - since a cancellation is a different patient behavior from a silent no-show, I'll need to decide (together with the Data Science track) whether to treat it separately, group it with no-shows, or exclude it from certain comparisons.
5. **Build a small number of clear charts/dashboard visuals** - once the KPIs are calculated, rather than a large number of charts, so the results stay easy for clinic staff to act on.
6. **Share findings with other tracks** - since this analysis will help inform what the Data Science track uses to build a prediction model, and what the Generative AI track needs to know about common patient issues (like reminders and rescheduling).

## Assumptions, Limitations, and Risks

**Assumptions**
- I'm assuming this dataset reflects realistic appointment patterns, even though it's a fictional/simulated dataset for this training program.
- I'm assuming that a missing `reminder_channel` simply means no reminder was sent, not that data was lost or entered incorrectly.

**Limitations**
- The dataset doesn't include information like income, insurance/payment status, or how serious a patient's condition is; these are common real-world reasons patients miss appointments, but I can't measure them here.
- Some appointment dates go into 2026, which is later than expected; this is likely just how the fictional data was generated, so I'll be cautious about concluding based on time trends.
- A small number of rows are missing distance and waiting time values (1-2%), so I'll need to decide whether to exclude these rows or fill them in before calculating final KPIs.
- Only about 5% of appointments are "Cancelled," which is a small group; I need to be careful not to draw strong conclusions from such a small sample.

**Risks**
- Any definitions I use here (like what counts as a "no-show" vs. a "cancellation") need to stay consistent with what the Data Science and Generative AI tracks are using, or the final combined project could contradict itself.
- The patterns I've noticed so far (like reminders being linked to fewer no-shows) are just observations, not proof of cause and effect; I should be careful not to present them as guaranteed facts to clinic stakeholders.